# 🚀 PropIQ — Week 3B: Convert the Databricks Method

Use PropIQ to learn the method, then convert it to the assigned project.

The converted notebook must stop at:

```text
one project Bronze demo table
→ one project lineage demo view
```

The full project Bronze layer belongs to Week 4.

## 🎯 Week-3 outcome

By the end of this notebook, every intern should be able to:

- find uploaded files in a Databricks Volume;
- understand how Spark SQL reads CSV and JSON files;
- create temporary views;
- inspect table structure and column types;
- display table contents;
- explain grain and business keys;
- compare physical rows with distinct business records;
- inspect values and distributions;
- identify simple data-quality concerns;
- check relationships between files;
- create one small managed Delta preview;
- explain what belongs to Week 3 and what belongs to Week 4.

> **The objective is understanding—not speed.**

## 🧩 Databricks cell languages

Databricks allows different cell languages inside one notebook.

Use the cell-language dropdown to choose:

- **Python** for PySpark;
- **SQL** for Spark SQL;
- **File system** for `%fs` commands.

You can also place a magic command at the top of a cell:

```text
%python
%sql
%fs
```

In this notebook:

- PySpark is used for loading files and basic DataFrame inspection;
- Spark SQL is used for most exploration and analysis;
- selected activities are shown in both styles so interns can compare them.

## 🧭 Notebook map

| Section | What you will do |
|---|---|
| 1 | Check the uploaded files |
| 2 | Create Spark SQL views |
| 3 | Inspect schemas |
| 4 | Display table contents |
| 5 | Understand grain |
| 6 | Count records |
| 7 | Inspect values |
| 8 | Find simple data concerns |
| 9 | Check relationships |
| 10 | Ask one business question |
| 11 | Create a Bronze preview |
| 12 | Capture evidence and explain the work |

# 1. Prepare Databricks

Before running this notebook:

1. Open your Databricks workspace.
2. Attach **Serverless notebook compute**.
3. Create a Volume named `propiq`.
4. Upload these three Week-3 files:

```text
listings.csv.gz
leads.csv
brokers.json
```

Recommended location:

```text
/Volumes/workspace/default/propiq/
```

Week-10 event files are not required today.

## 💡 Tip — Know where things live

| Item | Correct place |
|---|---|
| Notebook | Databricks Workspace |
| Full data files | Unity Catalog Volume |
| Screenshots | GitHub repository |
| Weekly log | GitHub repository |
| Full working dataset | Do not commit to GitHub |

A notebook contains instructions.

A Volume contains data.

# 2. Check the uploaded files

Before loading data, confirm that the files are visible.

The following command lists the contents of the PropIQ Volume.

In [0]:
%fs
ls /Volumes/workspace/default/propiq

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/propiq/brokers.csv,brokers.csv,40428,1784738115000
dbfs:/Volumes/workspace/default/propiq/leads.csv,leads.csv,17096212,1784738237000
dbfs:/Volumes/workspace/default/propiq/listings.parquet,listings.parquet,2533387,1784738145000
dbfs:/Volumes/workspace/default/propiq/localities.json,localities.json,24420,1784738115000


### Expected files

```text
listings.csv.gz
leads.csv
brokers.json
```

If one is missing, stop and upload it before continuing.

> **Professional habit:** Always confirm the source files before writing queries.

# 3. Create Spark SQL views

A Spark SQL view gives a file a simple table-like name.

Instead of repeatedly referring to a long file path, we can write:

```sql
SELECT * FROM listings
```

We will create one temporary view for each source file.

## 3.1 Create the `listings` view

The loan file is a compressed CSV.

`header = true` tells Spark that the first row contains column names.

`inferSchema = true` asks Spark to detect common data types.

### What this Python block does

This cell:

1. reads the compressed CSV file;
2. creates a PySpark DataFrame named `listings`;
3. creates a temporary SQL view named `listings`.

Use the cell-language dropdown and select **Python** before running it.

In [0]:
# Load the loan file as a PySpark DataFrame
listings = spark.read.parquet(
    "/Volumes/workspace/default/propiq/listings.parquet",
    header=True,
    inferSchema=True
)

# Make the DataFrame available to Spark SQL
listings.createOrReplaceTempView("listings")

## 3.2 Create the `leads` view

### What this Python block does

This cell reads the leads CSV and creates both:

- a PySpark DataFrame named `leads`;
- a Spark SQL temporary view named `leads`.

In [0]:
# Load the leads file
leads = spark.read.csv(
    "/Volumes/workspace/default/propiq/leads.csv",
    header=True,
    inferSchema=True
)

# Make it available to Spark SQL
leads.createOrReplaceTempView("leads")

## 3.3 Create the `brokers` view

### What this Python block does

This cell reads the JSON branch file and creates both:

- a PySpark DataFrame named `brokers`;
- a Spark SQL temporary view named `brokers`.

In [0]:
# Load the brokers file as a PySpark DataFrame

brokers = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv("/Volumes/workspace/default/propiq/brokers.csv")
)

# Make the DataFrame available to Spark SQL

brokers.createOrReplaceTempView("brokers")

display(brokers)

broker_id,agency_name,city,broker_tier,active_flag,onboarded_date,service_rating,source_system,batch_id,record_uid
BRK-0001,PropIQ Synthetic Agency 001,Hyderabad,Standard,false,2021-01-01,3.0,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000001
BRK-0002,PropIQ Synthetic Agency 002,Bengaluru,Verified,true,2021-01-20,4.7,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000002
BRK-0003,PropIQ Synthetic Agency 003,Pune,Premier,true,2021-02-08,4.4,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000003
BRK-0004,PropIQ Synthetic Agency 004,Chennai,Standard,true,2021-02-27,4.1,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000004
BRK-0005,PropIQ Synthetic Agency 005,Hyderabad,Verified,true,2021-03-18,3.8,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000005
BRK-0006,PropIQ Synthetic Agency 006,Bengaluru,Premier,true,2021-04-06,3.5,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000006
BRK-0007,PropIQ Synthetic Agency 007,Pune,Standard,true,2021-04-25,3.2,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000007
BRK-0008,PropIQ Synthetic Agency 008,Chennai,Verified,true,2021-05-14,4.9,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000008
BRK-0009,PropIQ Synthetic Agency 009,Hyderabad,Premier,true,2021-06-02,4.6,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000009
BRK-0010,PropIQ Synthetic Agency 010,Bengaluru,Standard,true,2021-06-21,4.3,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000010


## 3.4 Confirm that the views exist

In [0]:
%sql
SHOW TABLES;

database,tableName,isTemporary
default,bronze_demo_listings,false
default,listings_delta,false
default,propiq_week03_bronze_demo_listings,false
default,propiq_week03_lineage_demo_view,false
,brokers,true
,leads,true
,listings,true


You should see temporary views named:

```text
listings
leads
brokers
```

These views exist for the current notebook session.

# 3A. Confirm the created DataFrames

At this point, three PySpark DataFrames should exist:

```text
listings
leads
brokers
```

Use the following short Python cell to display their names and column counts.

In [0]:
# Confirm the three DataFrames and their column counts
print("listings columns:", len(listings.columns))
print("leads columns:", len(leads.columns))
print("brokers columns:", len(brokers.columns))

listings columns: 17
leads columns: 12
brokers columns: 10


This is a simple existence check.

It does not replace schema inspection, row counts or table previews.

# 4. Inspect the schema

A schema describes the structure of a dataset.

It tells us:

- column names;
- data types;
- possible identifiers;
- date and timestamp fields;
- numeric measures;
- nullable fields.

We will inspect each view separately.

## 4.1 Loans schema

### PySpark method — print the DataFrame schema

This is the quickest way to inspect DataFrame columns and data types.

In [0]:
# Show loan column names and data types
listings.printSchema()

root
 |-- record_uid: string (nullable = true)
 |-- listing_id: string (nullable = true)
 |-- locality_id: string (nullable = true)
 |-- broker_id: string (nullable = true)
 |-- property_type: string (nullable = true)
 |-- bedrooms: long (nullable = true)
 |-- furnishing: string (nullable = true)
 |-- built_up_area_sqft: long (nullable = true)
 |-- asking_price_inr: long (nullable = true)
 |-- price_per_sqft: long (nullable = true)
 |-- listing_created_date: string (nullable = true)
 |-- last_updated_timestamp: string (nullable = true)
 |-- completion_date: string (nullable = true)
 |-- listing_status: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_record_id: string (nullable = true)
 |-- batch_id: string (nullable = true)



### Spark SQL method — describe the SQL view

The SQL version displays the same structure in table form.

In [0]:
%sql
DESCRIBE listings;

col_name,data_type,comment
record_uid,string,null
listing_id,string,null
locality_id,string,null
broker_id,string,null
property_type,string,null
bedrooms,bigint,null
furnishing,string,null
built_up_area_sqft,bigint,null
asking_price_inr,bigint,null
price_per_sqft,bigint,null


### What to notice

Look for:

- `listing_id` — likely business key;
- `member_code` — member identifier;
- `lead_id` and `broker_id` — relationship fields;
- `checkout_ts`, `due_ts`, `return_ts` — time fields;
- `status` — category field;
- `loan_period_days`, `fine_amount`, `net_fine` — numeric fields.

## 4.2 Books schema

In [0]:
%sql
DESCRIBE leads;

col_name,data_type,comment
record_uid,string,null
lead_id,string,null
listing_id,string,null
lead_timestamp,timestamp,null
lead_channel,string,null
buyer_intent,string,null
qualified_flag,boolean,null
lead_status,string,null
budget_band,string,null
source_system,string,null


## 4.3 Branches schema

In [0]:
%sql
DESCRIBE brokers;

col_name,data_type,comment
broker_id,string,null
agency_name,string,null
city,string,null
broker_tier,string,null
active_flag,boolean,null
onboarded_date,date,null
service_rating,double,null
source_system,string,null
batch_id,string,null
record_uid,string,null


## 💡 Tip — Compare with Week 2

Open the Week-2 data dictionary.

Compare it manually with the actual schema.

Ask:

- Are the expected columns present?
- Did Spark detect the expected data types?
- Is any column missing?
- Is any extra column present?
- Does the proposed business key actually exist?

Do not automate this comparison yet.

First learn to read the schema yourself.

# 5. Display the table contents

A schema tells us the structure.

The actual rows tell us how the data looks.

Always inspect a few rows before writing analytical queries.

## 5.1 Display loan records

### PySpark method — display DataFrame rows

This uses the `listings` DataFrame created earlier.

In [0]:
# Display the first 10 loan records
display(listings.limit(10))

record_uid,listing_id,locality_id,broker_id,property_type,bedrooms,furnishing,built_up_area_sqft,asking_price_inr,price_per_sqft,listing_created_date,last_updated_timestamp,completion_date,listing_status,source_system,source_record_id,batch_id
LISTING-PHY-0000001,LST-0000001,LOC-062,BRK-0191,Apartment,4,Furnished,1631,11005988,6748,2025-05-15,2025-09-03T21:21:36,null,active,BROKER_CRM,SRC-L-0000001,BATCH-2026-01
LISTING-PHY-0000002,LST-0000002,LOC-056,BRK-0130,Apartment,2,Semi-furnished,1157,21793252,18836,2025-07-17,2025-08-10T14:05:46,null,active,BROKER_CRM,SRC-L-0000002,BATCH-2026-01
LISTING-PHY-0000003,LST-0000003,LOC-045,BRK-0215,Apartment,1,Semi-furnished,1973,16873096,8552,2024-12-23,2025-01-02T13:35:06,2025-03-01,rented,AGENCY_UPLOAD,SRC-L-0000003,BATCH-2026-01
LISTING-PHY-0000004,LST-0000004,LOC-036,BRK-0177,Apartment,4,Furnished,1164,21144060,18165,2024-10-31,2024-11-08T16:45:22,null,paused,PORTAL_FEED_A,SRC-L-0000004,BATCH-2026-01
LISTING-PHY-0000005,LST-0000005,LOC-041,BRK-0298,Apartment,3,Furnished,1478,9571528,6476,2025-07-28,2025-10-17T21:50:45,2025-12-04,sold,BROKER_CRM,SRC-L-0000005,BATCH-2026-01
LISTING-PHY-0000006,LST-0000006,LOC-014,BRK-0050,Apartment,3,Unfurnished,1827,10529001,5763,2024-05-25,2024-08-28T03:01:36,2024-09-26,sold,PORTAL_FEED_A,SRC-L-0000006,BATCH-2026-01
LISTING-PHY-0000007,LST-0000007,LOC-029,BRK-0055,Villa,4,Semi-furnished,4740,34199100,7215,2025-10-02,2025-11-22T06:51:33,null,expired,PORTAL_FEED_A,SRC-L-0000007,BATCH-2026-01
LISTING-PHY-0000008,LST-0000008,LOC-074,BRK-0011,Apartment,1,Semi-furnished,1296,12205728,9418,2024-01-30,2024-05-18T02:03:22,null,active,AGENCY_UPLOAD,SRC-L-0000008,BATCH-2026-01
LISTING-PHY-0000009,LST-0000009,LOC-046,BRK-0017,Apartment,1,Unfurnished,1301,9395822,7222,2025-02-04,2025-06-11T17:21:51,2025-06-09,rented,BROKER_CRM,SRC-L-0000009,BATCH-2026-01
LISTING-PHY-0000010,LST-0000010,LOC-071,BRK-0201,Villa,2,Semi-furnished,3882,39615810,10205,2024-02-12,2024-04-30T07:39:38,2024-05-01,sold,PORTAL_FEED_A,SRC-L-0000010,BATCH-2026-01


### Spark SQL method — display the same rows

The SQL query below reads from the temporary view created from the same DataFrame.

In [0]:
%sql
SELECT *
FROM listings
LIMIT 10;

record_uid,listing_id,locality_id,broker_id,property_type,bedrooms,furnishing,built_up_area_sqft,asking_price_inr,price_per_sqft,listing_created_date,last_updated_timestamp,completion_date,listing_status,source_system,source_record_id,batch_id
LISTING-PHY-0000001,LST-0000001,LOC-062,BRK-0191,Apartment,4,Furnished,1631,11005988,6748,2025-05-15,2025-09-03T21:21:36,null,active,BROKER_CRM,SRC-L-0000001,BATCH-2026-01
LISTING-PHY-0000002,LST-0000002,LOC-056,BRK-0130,Apartment,2,Semi-furnished,1157,21793252,18836,2025-07-17,2025-08-10T14:05:46,null,active,BROKER_CRM,SRC-L-0000002,BATCH-2026-01
LISTING-PHY-0000003,LST-0000003,LOC-045,BRK-0215,Apartment,1,Semi-furnished,1973,16873096,8552,2024-12-23,2025-01-02T13:35:06,2025-03-01,rented,AGENCY_UPLOAD,SRC-L-0000003,BATCH-2026-01
LISTING-PHY-0000004,LST-0000004,LOC-036,BRK-0177,Apartment,4,Furnished,1164,21144060,18165,2024-10-31,2024-11-08T16:45:22,null,paused,PORTAL_FEED_A,SRC-L-0000004,BATCH-2026-01
LISTING-PHY-0000005,LST-0000005,LOC-041,BRK-0298,Apartment,3,Furnished,1478,9571528,6476,2025-07-28,2025-10-17T21:50:45,2025-12-04,sold,BROKER_CRM,SRC-L-0000005,BATCH-2026-01
LISTING-PHY-0000006,LST-0000006,LOC-014,BRK-0050,Apartment,3,Unfurnished,1827,10529001,5763,2024-05-25,2024-08-28T03:01:36,2024-09-26,sold,PORTAL_FEED_A,SRC-L-0000006,BATCH-2026-01
LISTING-PHY-0000007,LST-0000007,LOC-029,BRK-0055,Villa,4,Semi-furnished,4740,34199100,7215,2025-10-02,2025-11-22T06:51:33,null,expired,PORTAL_FEED_A,SRC-L-0000007,BATCH-2026-01
LISTING-PHY-0000008,LST-0000008,LOC-074,BRK-0011,Apartment,1,Semi-furnished,1296,12205728,9418,2024-01-30,2024-05-18T02:03:22,null,active,AGENCY_UPLOAD,SRC-L-0000008,BATCH-2026-01
LISTING-PHY-0000009,LST-0000009,LOC-046,BRK-0017,Apartment,1,Unfurnished,1301,9395822,7222,2025-02-04,2025-06-11T17:21:51,2025-06-09,rented,BROKER_CRM,SRC-L-0000009,BATCH-2026-01
LISTING-PHY-0000010,LST-0000010,LOC-071,BRK-0201,Villa,2,Semi-furnished,3882,39615810,10205,2024-02-12,2024-04-30T07:39:38,2024-05-01,sold,PORTAL_FEED_A,SRC-L-0000010,BATCH-2026-01


### Look carefully

Notice:

- how IDs are formatted;
- how timestamps appear;
- whether `return_ts` can be empty;
- the values used in `status`;
- whether numeric fields contain decimals or negatives.

## 5.2 Display book records

### PySpark method — display the leads DataFrame

This confirms that the CSV was loaded into the `leads` DataFrame.

In [0]:
# Display the first 10 book records
display(leads.limit(10))

record_uid,lead_id,listing_id,lead_timestamp,lead_channel,buyer_intent,qualified_flag,lead_status,budget_band,source_system,source_record_id,batch_id
LEAD-PHY-0000001,LED-00000001,LST-0018000,2026-05-25T18:19:50.000Z,Campaign,Medium,false,new,2Cr-5Cr,CAMPAIGN,SRC-D-00000001,BATCH-2026-01
LEAD-PHY-0000002,LED-00000002,LST-0024708,2025-10-22T21:10:21.000Z,Portal Search,Medium,true,qualified,1Cr-2Cr,MOBILE,SRC-D-00000002,BATCH-2026-01
LEAD-PHY-0000003,LED-00000003,LST-0020315,2026-04-17T01:24:48.000Z,Portal Search,High,false,new,Under 50L,BROKER_CRM,SRC-D-00000003,BATCH-2026-01
LEAD-PHY-0000004,LED-00000004,LST-0015485,2025-03-01T21:32:16.000Z,Partner,Medium,true,negotiation,1Cr-2Cr,MOBILE,SRC-D-00000004,BATCH-2026-01
LEAD-PHY-0000005,LED-00000005,LST-0037002,2025-06-18T04:22:06.000Z,Portal Search,Medium,true,qualified,1Cr-2Cr,MOBILE,SRC-D-00000005,BATCH-2026-01
LEAD-PHY-0000006,LED-00000006,LST-0019134,2026-02-14T18:22:41.000Z,Portal Search,Exploratory,true,negotiation,50L-1Cr,BROKER_CRM,SRC-D-00000006,BATCH-2026-01
LEAD-PHY-0000007,LED-00000007,LST-0009732,2026-05-17T07:00:28.000Z,Portal Search,Medium,true,site_visit,1Cr-2Cr,CAMPAIGN,SRC-D-00000007,BATCH-2026-01
LEAD-PHY-0000008,LED-00000008,LST-0004899,2026-04-18T01:45:58.000Z,Portal Search,Medium,false,closed_unqualified,50L-1Cr,WEB,SRC-D-00000008,BATCH-2026-01
LEAD-PHY-0000009,LED-00000009,LST-0005907,2026-02-24T22:13:22.000Z,Walk-in,Exploratory,true,negotiation,50L-1Cr,WEB,SRC-D-00000009,BATCH-2026-01
LEAD-PHY-0000010,LED-00000010,LST-0017045,2025-02-27T15:19:28.000Z,Portal Search,Exploratory,true,qualified,Under 50L,BROKER_CRM,SRC-D-00000010,BATCH-2026-01


### Spark SQL method — display the leads view

The SQL query below reads the temporary view created from the same DataFrame.

In [0]:
%sql
SELECT *
FROM leads
LIMIT 10;

record_uid,lead_id,listing_id,lead_timestamp,lead_channel,buyer_intent,qualified_flag,lead_status,budget_band,source_system,source_record_id,batch_id
LEAD-PHY-0000001,LED-00000001,LST-0018000,2026-05-25T18:19:50.000Z,Campaign,Medium,false,new,2Cr-5Cr,CAMPAIGN,SRC-D-00000001,BATCH-2026-01
LEAD-PHY-0000002,LED-00000002,LST-0024708,2025-10-22T21:10:21.000Z,Portal Search,Medium,true,qualified,1Cr-2Cr,MOBILE,SRC-D-00000002,BATCH-2026-01
LEAD-PHY-0000003,LED-00000003,LST-0020315,2026-04-17T01:24:48.000Z,Portal Search,High,false,new,Under 50L,BROKER_CRM,SRC-D-00000003,BATCH-2026-01
LEAD-PHY-0000004,LED-00000004,LST-0015485,2025-03-01T21:32:16.000Z,Partner,Medium,true,negotiation,1Cr-2Cr,MOBILE,SRC-D-00000004,BATCH-2026-01
LEAD-PHY-0000005,LED-00000005,LST-0037002,2025-06-18T04:22:06.000Z,Portal Search,Medium,true,qualified,1Cr-2Cr,MOBILE,SRC-D-00000005,BATCH-2026-01
LEAD-PHY-0000006,LED-00000006,LST-0019134,2026-02-14T18:22:41.000Z,Portal Search,Exploratory,true,negotiation,50L-1Cr,BROKER_CRM,SRC-D-00000006,BATCH-2026-01
LEAD-PHY-0000007,LED-00000007,LST-0009732,2026-05-17T07:00:28.000Z,Portal Search,Medium,true,site_visit,1Cr-2Cr,CAMPAIGN,SRC-D-00000007,BATCH-2026-01
LEAD-PHY-0000008,LED-00000008,LST-0004899,2026-04-18T01:45:58.000Z,Portal Search,Medium,false,closed_unqualified,50L-1Cr,WEB,SRC-D-00000008,BATCH-2026-01
LEAD-PHY-0000009,LED-00000009,LST-0005907,2026-02-24T22:13:22.000Z,Walk-in,Exploratory,true,negotiation,50L-1Cr,WEB,SRC-D-00000009,BATCH-2026-01
LEAD-PHY-0000010,LED-00000010,LST-0017045,2025-02-27T15:19:28.000Z,Portal Search,Exploratory,true,qualified,Under 50L,BROKER_CRM,SRC-D-00000010,BATCH-2026-01


## 5.3 Display branch records

### PySpark method — display the brokers DataFrame

This confirms that the JSON file was loaded into the `brokers` DataFrame.

In [0]:
# Display the first 10 branch records
display(brokers.orderBy("broker_id").limit(10))

broker_id,agency_name,city,broker_tier,active_flag,onboarded_date,service_rating,source_system,batch_id,record_uid
BRK-0001,PropIQ Synthetic Agency 001,Hyderabad,Standard,false,2021-01-01,3.0,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000001
BRK-0002,PropIQ Synthetic Agency 002,Bengaluru,Verified,true,2021-01-20,4.7,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000002
BRK-0003,PropIQ Synthetic Agency 003,Pune,Premier,true,2021-02-08,4.4,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000003
BRK-0004,PropIQ Synthetic Agency 004,Chennai,Standard,true,2021-02-27,4.1,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000004
BRK-0005,PropIQ Synthetic Agency 005,Hyderabad,Verified,true,2021-03-18,3.8,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000005
BRK-0006,PropIQ Synthetic Agency 006,Bengaluru,Premier,true,2021-04-06,3.5,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000006
BRK-0007,PropIQ Synthetic Agency 007,Pune,Standard,true,2021-04-25,3.2,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000007
BRK-0008,PropIQ Synthetic Agency 008,Chennai,Verified,true,2021-05-14,4.9,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000008
BRK-0009,PropIQ Synthetic Agency 009,Hyderabad,Premier,true,2021-06-02,4.6,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000009
BRK-0010,PropIQ Synthetic Agency 010,Bengaluru,Standard,true,2021-06-21,4.3,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000010


### Spark SQL method — display the brokers view

The SQL query below reads the temporary view created from the same DataFrame.

In [0]:
%sql
SELECT *
FROM brokers
ORDER BY broker_id
LIMIT 10;

broker_id,agency_name,city,broker_tier,active_flag,onboarded_date,service_rating,source_system,batch_id,record_uid
BRK-0001,PropIQ Synthetic Agency 001,Hyderabad,Standard,false,2021-01-01,3.0,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000001
BRK-0002,PropIQ Synthetic Agency 002,Bengaluru,Verified,true,2021-01-20,4.7,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000002
BRK-0003,PropIQ Synthetic Agency 003,Pune,Premier,true,2021-02-08,4.4,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000003
BRK-0004,PropIQ Synthetic Agency 004,Chennai,Standard,true,2021-02-27,4.1,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000004
BRK-0005,PropIQ Synthetic Agency 005,Hyderabad,Verified,true,2021-03-18,3.8,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000005
BRK-0006,PropIQ Synthetic Agency 006,Bengaluru,Premier,true,2021-04-06,3.5,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000006
BRK-0007,PropIQ Synthetic Agency 007,Pune,Standard,true,2021-04-25,3.2,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000007
BRK-0008,PropIQ Synthetic Agency 008,Chennai,Verified,true,2021-05-14,4.9,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000008
BRK-0009,PropIQ Synthetic Agency 009,Hyderabad,Premier,true,2021-06-02,4.6,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000009
BRK-0010,PropIQ Synthetic Agency 010,Bengaluru,Standard,true,2021-06-21,4.3,PROPIQ_SYNTH_MASTER,BATCH-2026-01,BROKER-PHY-000010


## 🧠 Intern checkpoint 1

Complete these statements:

```text
The main transaction file is ____________________.
One row appears to represent ____________________.
The likely business key is ____________________.
The main date field is ____________________.
The main status field is ____________________.
```

Do not move ahead until the answers make sense.

# 6. Understand the grain

## What is grain?

**Grain means what one row represents.**

Examples:

- one loan record;
- one book;
- one branch;
- one payment;
- one delivery;
- one incident.

For PropIQ:

| View | Expected grain |
|---|---|
| `listings` | one physical loan source record |
| `leads` | one book record |
| `brokers` | one branch record |

The grain must be understood before counting business activity.

# 7. Count the physical records

Start with a simple row count for each view.

## PySpark method — count one DataFrame

This counts the physical rows in the main loan DataFrame.

In [0]:
# Count physical loan rows
loan_row_count = listings.count()
print("Physical loan rows:", loan_row_count)

Physical loan rows: 50200


## Spark SQL method — count all three views

The SQL query below produces a compact source summary.

In [0]:
%sql
SELECT 'listings' AS source, COUNT(*) AS records
FROM listings
UNION ALL
SELECT 'leads', COUNT(*)
FROM leads
UNION ALL
SELECT 'brokers', COUNT(*)
FROM brokers;

source,records
listings,50200
leads,120800
brokers,320


This query tells us how many physical rows Spark loaded from each source.

For the loan file, the expected physical row count is:

```text
90,279
```

# 8. Compare rows with distinct business keys

A physical row count is not always the same as a business record count.

The proposed loan business key is `listing_id`.

Let us count distinct loan IDs.

## PySpark method — count distinct business keys

This uses simple DataFrame operations to count unique `listing_id` values.

In [0]:
# Count distinct loan IDs
distinct_loan_count = (
    listings.select("listing_id")
         .distinct()
         .count()
)

print("Distinct loan IDs:", distinct_loan_count)

Distinct loan IDs: 49901


## Spark SQL method — compare both values together

The SQL query below is more compact for analytical comparison.

In [0]:
%sql
SELECT
  COUNT(*) AS physical_rows,
  COUNT(DISTINCT listing_id) AS distinct_listings
FROM listings;

physical_rows,distinct_listings
50200,49901


### Expected PropIQ result

```text
Physical rows:      90,279
Distinct loan IDs:  90,000
Difference:            279
```

This means the file contains repeated business keys.

> **Professional interpretation:** There are 90,279 source records representing 90,000 distinct listings.

# 9. Display repeated business keys

Now identify a few repeated `listing_id` values.

In [0]:
%sql
SELECT
  listing_id,
  COUNT(*) AS occurrences
FROM listings
GROUP BY listing_id
HAVING COUNT(*) > 1
ORDER BY occurrences DESC
LIMIT 20;

listing_id,occurrences
,100
LST-0047301,2
LST-0045294,2
LST-0048646,2
LST-0046754,2
LST-0047154,2
LST-0046325,2
LST-0047868,2
LST-0048920,2
LST-0048313,2


## 🧠 Intern checkpoint 2

Explain this in one sentence:

```text
The row count is higher than the distinct-loan count because
____________________________________________________________.
```

This is the first grain-related discovery.

# 10. Inspect important values

Before looking for errors, understand the normal values in the file.

## 10.1 Status distribution

### PySpark method — group and count

This groups the DataFrame by `status` and counts records.

In [0]:
display(
    listings.groupBy("listing_status")
            .count()
            .orderBy("count", ascending=False)
)

listing_status,count
active,23921
sold,8614
rented,5563
expired,5011
paused,3996
withdrawn,3095


### Spark SQL method — perform the same analysis

The SQL version below gives the same business result.

In [0]:
%sql
SELECT
    listing_status,
    COUNT(*) AS records
FROM listings
GROUP BY listing_status
ORDER BY records DESC;

listing_status,records
active,23921
sold,8614
rented,5563
expired,5011
paused,3996
withdrawn,3095


### Why this matters

A distribution helps us understand:

- common categories;
- rare categories;
- possible spelling differences;
- unexpected values;
- whether one category dominates the data.

## 10.2 Checkout date range

In [0]:
%sql
SELECT
    MIN(listing_created_date) AS earliest_listing,
    MAX(listing_created_date) AS latest_listing
FROM listings;

earliest_listing,latest_listing
2024-01-01,2026-03-30


## 10.3 Loan-period range

In [0]:
%sql
SELECT
    MIN(asking_price_inr) AS minimum_price,
    MAX(asking_price_inr) AS maximum_price,
    AVG(asking_price_inr) AS average_price
FROM listings;

minimum_price,maximum_price,average_price
-250000,132265650,1.790185924378486E7


## 10.4 Fine-value range

In [0]:
%sql
SELECT
    MIN(built_up_area_sqft) AS minimum_area,
    MAX(built_up_area_sqft) AS maximum_area,
    AVG(built_up_area_sqft) AS average_area
FROM listings;

minimum_area,maximum_area,average_area
120,7291,1694.7809561752988


# 11. Find simple data concerns

Week 3 is about observation.

We are not cleaning the data yet.

We are only asking:

> **What should the Week-4 and Week-5 pipeline handle?**

## 11.1 Missing member codes

In [0]:
%sql
SELECT
    COUNT(*) AS missing_broker_ids
FROM listings
WHERE broker_id IS NULL;

missing_broker_ids
0


## 11.2 Negative loan periods

In [0]:
%sql
SELECT
    COUNT(*) AS invalid_property_area
FROM listings
WHERE built_up_area_sqft <= 0;

invalid_property_area
0


Expected result:

```text
180 records
```

A negative loan period is logically suspicious.

## 11.3 Return before checkout

In [0]:
%sql
SELECT
    COUNT(*) AS future_listing_dates
FROM listings
WHERE listing_created_date > CURRENT_DATE();

future_listing_dates
0


Expected result:
Run and record actual result.

Interpretation:
Listings with future dates indicate data quality issues that should be investigated before downstream processing.

## 11.4 Display a few suspicious records

In [0]:
%sql
SELECT
    listing_id,
    broker_id,
    locality_id,
    property_type,
    asking_price_inr,
    bedrooms,
    listing_status
FROM listings
WHERE asking_price_inr <= 0
   OR bedrooms < 0
LIMIT 20;

listing_id,broker_id,locality_id,property_type,asking_price_inr,bedrooms,listing_status
LST-0045089,BRK-0261,LOC-069,Apartment,-250000,2,sold
LST-0045112,BRK-0162,LOC-057,Apartment,-250000,1,active
LST-0045134,BRK-0177,LOC-034,Row House,-250000,2,active
LST-0045289,BRK-0318,LOC-060,Apartment,-250000,5,active
LST-0045314,BRK-0100,LOC-079,Row House,-250000,3,sold
LST-0045320,BRK-0147,LOC-017,Apartment,-250000,4,active
LST-0045395,BRK-0234,LOC-027,Apartment,-250000,1,active
LST-0045399,BRK-0194,LOC-031,Apartment,-250000,3,rented
LST-0045431,BRK-0235,LOC-023,Apartment,-250000,1,expired
LST-0045483,BRK-0099,LOC-054,Villa,-250000,5,expired


## 🧠 Intern checkpoint 3

Choose one issue and explain:

```text
Issue:
Why it matters:
Which later week should handle it:
```

Suggested answer structure:

> “I found ________. It could affect ________. It should be handled during Silver or Data Quality work.”

# 12. Check relationships between files

The loan file contains:

- `lead_id`;
- `broker_id`.

A good relationship means the referenced value exists in the related file.

## 12.1 Check the book relationship

In [0]:
%sql
SELECT
    COUNT(*) AS invalid_listing_references
FROM leads ld
LEFT JOIN listings l
ON ld.listing_id = l.listing_id
WHERE l.listing_id IS NULL;

invalid_listing_references
600


Expected result:
Run and record actual result.

Interpretation:
These lead records reference listing IDs that do not exist in the listings dataset and should be investigated before Bronze ingestion.

## 12.2 Display a few invalid book references

In [0]:
%sql
SELECT
    ld.lead_id,
    ld.listing_id,
    ld.lead_status,
    ld.lead_channel
FROM leads ld
LEFT JOIN listings l
ON ld.listing_id = l.listing_id
WHERE l.listing_id IS NULL
LIMIT 20;

lead_id,listing_id,lead_status,lead_channel
LED-00000088,LST-9999999,contacted,Partner
LED-00000725,LST-9999999,site_visit,Partner
LED-00001161,LST-9999999,negotiation,Portal Search
LED-00001486,LST-9999999,qualified,Portal Search
LED-00001487,LST-9999999,new,Campaign
LED-00001636,LST-9999999,negotiation,Portal Search
LED-00001736,LST-9999999,qualified,Portal Search
LED-00002047,LST-9999999,closed_unqualified,Portal Search
LED-00002272,LST-9999999,site_visit,Portal Search
LED-00002372,LST-9999999,new,Campaign


## 12.3 Check the branch relationship

In [0]:
%sql
SELECT COUNT(*) AS invalid_branch_references
FROM listings l
LEFT JOIN brokers b
  ON l.broker_id = b.broker_id
WHERE b.broker_id IS NULL;

invalid_branch_references
100


Expected result:

```text
100 invalid branch references
```

Zero is still valuable evidence because the relationship was tested.

# 13. Understand the join effect

An inner join keeps only matching records.

If 558 loan rows do not find a book, those rows will not survive an inner join.

In [0]:
%sql
SELECT
    COUNT(*) AS matched_lead_records
FROM leads ld
INNER JOIN listings l
ON ld.listing_id = l.listing_id;

matched_lead_records
120200


Expected result:
Run and record actual result.

Interpretation:
The INNER JOIN returns only lead records that reference an existing listing.
Any unmatched leads are excluded.

# 14. Ask one business question

The library manager asks:

> Which branch zone has the highest raw loan activity?

We will join listings with brokers and count the records.

In [0]:
%sql
SELECT
    property_type,
    COUNT(*) AS total_listings,
    COUNT(DISTINCT listing_id) AS distinct_listings
FROM listings
GROUP BY property_type
ORDER BY total_listings DESC;

property_type,total_listings,distinct_listings
Apartment,34452,34248
Villa,6960,6930
Row House,5092,5053
Studio,3546,3523
Unknown Tower,150,150


Highest property type:
Total listings:
Distinct listing IDs:
Observation:
Run and record actual result after executing the query.

# 15. Preview the Bronze idea

The Week-3 flow is:

```text
Volume files
→ PySpark DataFrames
→ temporary Spark SQL views
→ exploration
→ one Bronze demonstration table
→ one lineage demonstration view
```

Only one Bronze demonstration table is created here. The complete multi-table Bronze layer belongs to Week 4.

## 15.1 Create one Bronze demo table

This managed Delta table uses the main `listings` entity, preserves the source columns and adds only basic ingestion metadata.

No deduplication, correction or Silver transformation is performed.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.propiq_week03_bronze_demo_listings
USING DELTA
AS
SELECT
  *,
  current_timestamp() AS ingested_at,
  '/Volumes/workspace/default/propiq/listings.csv.gz' AS source_file
FROM listings;

num_affected_rows,num_inserted_rows


> This is a Week-3 learning table—not the official PropIQ Bronze layer.

# 16. Confirm and display the demo table

In [0]:
%sql
SHOW TABLES IN workspace.default LIKE 'propiq_week03_bronze_demo_listings';

database,tableName,isTemporary
default,propiq_week03_bronze_demo_listings,false


In [0]:
%sql
SELECT *
FROM workspace.default.propiq_week03_bronze_demo_listings
LIMIT 10;

record_uid,listing_id,locality_id,broker_id,property_type,bedrooms,furnishing,built_up_area_sqft,asking_price_inr,price_per_sqft,listing_created_date,last_updated_timestamp,completion_date,listing_status,source_system,source_record_id,batch_id,ingested_at,source_file
LISTING-PHY-0000001,LST-0000001,LOC-062,BRK-0191,Apartment,4,Furnished,1631,11005988,6748,2025-05-15,2025-09-03T21:21:36,null,active,BROKER_CRM,SRC-L-0000001,BATCH-2026-01,2026-07-25T06:44:25.231Z,/Volumes/workspace/default/propiq/listings.csv.gz
LISTING-PHY-0000002,LST-0000002,LOC-056,BRK-0130,Apartment,2,Semi-furnished,1157,21793252,18836,2025-07-17,2025-08-10T14:05:46,null,active,BROKER_CRM,SRC-L-0000002,BATCH-2026-01,2026-07-25T06:44:25.231Z,/Volumes/workspace/default/propiq/listings.csv.gz
LISTING-PHY-0000003,LST-0000003,LOC-045,BRK-0215,Apartment,1,Semi-furnished,1973,16873096,8552,2024-12-23,2025-01-02T13:35:06,2025-03-01,rented,AGENCY_UPLOAD,SRC-L-0000003,BATCH-2026-01,2026-07-25T06:44:25.231Z,/Volumes/workspace/default/propiq/listings.csv.gz
LISTING-PHY-0000004,LST-0000004,LOC-036,BRK-0177,Apartment,4,Furnished,1164,21144060,18165,2024-10-31,2024-11-08T16:45:22,null,paused,PORTAL_FEED_A,SRC-L-0000004,BATCH-2026-01,2026-07-25T06:44:25.231Z,/Volumes/workspace/default/propiq/listings.csv.gz
LISTING-PHY-0000005,LST-0000005,LOC-041,BRK-0298,Apartment,3,Furnished,1478,9571528,6476,2025-07-28,2025-10-17T21:50:45,2025-12-04,sold,BROKER_CRM,SRC-L-0000005,BATCH-2026-01,2026-07-25T06:44:25.231Z,/Volumes/workspace/default/propiq/listings.csv.gz
LISTING-PHY-0000006,LST-0000006,LOC-014,BRK-0050,Apartment,3,Unfurnished,1827,10529001,5763,2024-05-25,2024-08-28T03:01:36,2024-09-26,sold,PORTAL_FEED_A,SRC-L-0000006,BATCH-2026-01,2026-07-25T06:44:25.231Z,/Volumes/workspace/default/propiq/listings.csv.gz
LISTING-PHY-0000007,LST-0000007,LOC-029,BRK-0055,Villa,4,Semi-furnished,4740,34199100,7215,2025-10-02,2025-11-22T06:51:33,null,expired,PORTAL_FEED_A,SRC-L-0000007,BATCH-2026-01,2026-07-25T06:44:25.231Z,/Volumes/workspace/default/propiq/listings.csv.gz
LISTING-PHY-0000008,LST-0000008,LOC-074,BRK-0011,Apartment,1,Semi-furnished,1296,12205728,9418,2024-01-30,2024-05-18T02:03:22,null,active,AGENCY_UPLOAD,SRC-L-0000008,BATCH-2026-01,2026-07-25T06:44:25.231Z,/Volumes/workspace/default/propiq/listings.csv.gz
LISTING-PHY-0000009,LST-0000009,LOC-046,BRK-0017,Apartment,1,Unfurnished,1301,9395822,7222,2025-02-04,2025-06-11T17:21:51,2025-06-09,rented,BROKER_CRM,SRC-L-0000009,BATCH-2026-01,2026-07-25T06:44:25.231Z,/Volumes/workspace/default/propiq/listings.csv.gz
LISTING-PHY-0000010,LST-0000010,LOC-071,BRK-0201,Villa,2,Semi-furnished,3882,39615810,10205,2024-02-12,2024-04-30T07:39:38,2024-05-01,sold,PORTAL_FEED_A,SRC-L-0000010,BATCH-2026-01,2026-07-25T06:44:25.231Z,/Volumes/workspace/default/propiq/listings.csv.gz


Look for:

```text
ingested_at
source_file
```

# 17. Perform one source-to-demo count check

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM listings) AS source_rows,
  (SELECT COUNT(*) FROM workspace.default.propiq_week03_bronze_demo_listings) AS demo_rows;

source_rows,demo_rows
50200,50200


Expected:

```text
source_rows = demo_rows
```

This is a simple Week-3 confidence check. Full reconciliation belongs to Week 4.

# 18. Inspect Delta table details

In [0]:
%sql
DESCRIBE DETAIL workspace.default.propiq_week03_bronze_demo_listings;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,ae6d2bdc-9683-44d8-8c7d-4de6f0870cb3,workspace.default.propiq_week03_bronze_demo_listings,null,,2026-07-25T06:44:24.935Z,2026-07-25T06:44:26.000Z,List(),List(),1,1007403,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


Notice fields such as `format`, `location`, `createdAt`, `lastModified` and `numFiles`.

# 19. Inspect Delta table history

In [0]:
%sql
DESCRIBE HISTORY workspace.default.propiq_week03_bronze_demo_listings;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-07-25T06:44:26.000Z,71736716437240,thota.madhulika05@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1530600452687422),aac0edda-c5fe-4a03-b8d7-96b7916bdf6a,0725-061709-trmfebt2-v2n,1,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 1007403, numDeletionVectorsRemoved -> 0, numOutputRows -> 50200, numOutputBytes -> 1007403)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
1,2026-07-25T06:39:55.000Z,71736716437240,thota.madhulika05@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1530600452687422),d142a2e4-58ad-4e28-9b09-1b66a7fd9e67,0725-061709-trmfebt2-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 1007403, numDeletionVectorsRemoved -> 0, numOutputRows -> 50200, numOutputBytes -> 1007403)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
0,2026-07-25T06:39:38.000Z,71736716437240,thota.madhulika05@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1530600452687422),028294eb-6689-46fb-9cfd-d1c18209e698,0725-061709-trmfebt2-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 50200, numOutputBytes -> 1007403)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13


| Concept | Question answered |
|---|---|
| Schema | What columns and data types exist? |
| Relationship | How do business entities connect? |
| History | What operations changed this Delta table? |
| Lineage | Which governed objects feed or use another object? |

# 20. Create a lineage demonstration view

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.default.propiq_week03_lineage_demo_view
AS
SELECT
    listing_id,
    record_uid,
    broker_id,
    locality_id,
    property_type,
    listing_status,
    asking_price_inr,
    bedrooms,
    built_up_area_sqft,
    listing_created_date,
    ingested_at
FROM workspace.default.propiq_week03_bronze_demo_listings;

In [0]:
%sql
SELECT *
FROM workspace.default.propiq_week03_lineage_demo_view
LIMIT 20;

listing_id,record_uid,broker_id,locality_id,property_type,listing_status,asking_price_inr,bedrooms,built_up_area_sqft,listing_created_date,ingested_at
LST-0000001,LISTING-PHY-0000001,BRK-0191,LOC-062,Apartment,active,11005988,4,1631,2025-05-15,2026-07-25T06:44:25.231Z
LST-0000002,LISTING-PHY-0000002,BRK-0130,LOC-056,Apartment,active,21793252,2,1157,2025-07-17,2026-07-25T06:44:25.231Z
LST-0000003,LISTING-PHY-0000003,BRK-0215,LOC-045,Apartment,rented,16873096,1,1973,2024-12-23,2026-07-25T06:44:25.231Z
LST-0000004,LISTING-PHY-0000004,BRK-0177,LOC-036,Apartment,paused,21144060,4,1164,2024-10-31,2026-07-25T06:44:25.231Z
LST-0000005,LISTING-PHY-0000005,BRK-0298,LOC-041,Apartment,sold,9571528,3,1478,2025-07-28,2026-07-25T06:44:25.231Z
LST-0000006,LISTING-PHY-0000006,BRK-0050,LOC-014,Apartment,sold,10529001,3,1827,2024-05-25,2026-07-25T06:44:25.231Z
LST-0000007,LISTING-PHY-0000007,BRK-0055,LOC-029,Villa,expired,34199100,4,4740,2025-10-02,2026-07-25T06:44:25.231Z
LST-0000008,LISTING-PHY-0000008,BRK-0011,LOC-074,Apartment,active,12205728,1,1296,2024-01-30,2026-07-25T06:44:25.231Z
LST-0000009,LISTING-PHY-0000009,BRK-0017,LOC-046,Apartment,rented,9395822,1,1301,2025-02-04,2026-07-25T06:44:25.231Z
LST-0000010,LISTING-PHY-0000010,BRK-0201,LOC-071,Villa,sold,39615810,2,3882,2024-02-12,2026-07-25T06:44:25.231Z


The governed lineage path is:

```text
propiq_week03_bronze_demo_listings
                 ↓
propiq_week03_lineage_demo_view
```

# 21. View lineage in Catalog Explorer

1. Click **Catalog**.
2. Open `workspace`.
3. Open `default`.
4. Select `propiq_week03_lineage_demo_view`.
5. Open **Lineage**.
6. Choose **See lineage graph** when available.
7. Identify `propiq_week03_bronze_demo_listings` as the upstream object.
8. Capture one screenshot.

## 🧠 Intern checkpoint

Explain:

```text
Files → DataFrames → temporary views → exploration
→ one Bronze demo table → one lineage demo view
```

Also explain why the complete Bronze layer is deferred to Week 4.

# 22. Week-3 boundary

## Completed

- source-file inspection;
- PySpark DataFrame creation and display;
- temporary SQL views;
- schema, grain, counts and values;
- simple data concerns;
- relationship checks;
- one business question;
- one Bronze demo table;
- one count check;
- Delta detail and history;
- one lineage demo view;
- Catalog Explorer lineage walkthrough.

## Deferred to Week 4

- official Bronze tables for every source;
- repeatable ingestion;
- complete reconciliation;
- schema handling policy;
- rerun behaviour;
- full Bronze evidence;
- production naming and controls.

# 23. Evidence checklist

```text
screenshots/week03_01_source_files.png
screenshots/week03_02_dataframes.png
screenshots/week03_03_schemas.png
screenshots/week03_04_grain_counts_values.png
screenshots/week03_05_relationship_checks.png
screenshots/week03_06_bronze_demo.png
screenshots/week03_07_delta_history.png
screenshots/week03_08_lineage_graph.png
```

# 24. Final intern defence

Every intern should explain:

1. Files, DataFrames and temporary views.
2. Schema, grain and business keys.
3. Physical rows versus distinct keys.
4. Values, ranges and data concerns.
5. Business relationships and joins.
6. What a managed Delta table is.
7. Why only one demo table was created.
8. What `DESCRIBE DETAIL` shows.
9. What `DESCRIBE HISTORY` shows.
10. What lineage means.
11. What is deferred to Week 4.

# 🎉 Week-3 Databricks foundation complete

```text
Volume files
→ PySpark DataFrames
→ temporary Spark SQL views
→ exploration
→ one Bronze demo table
→ one downstream lineage view
```

This is the correct Week-3 stopping point.

> Week 3 teaches how the data behaves. Week 4 builds the repeatable Bronze foundation.

# Part 2 — Project Conversion Studio

You have now seen the complete Week-3 method.

The next step is to convert:

```text
PropIQ files
PropIQ columns
PropIQ grain
PropIQ keys
PropIQ relationships
PropIQ question
```

into the real structure of your assigned project.

# 20. Prepare the conversion inputs

Provide the AI assistant with:

```text
1. This 03B notebook
2. Your project brief PDF
3. Your actual Data Pack ZIP
4. Your Week-2 data dictionary
```

The project title alone is not enough.

The AI must inspect the real files and fields.

# 21. Ask for a mapping before code

The AI must first produce a mapping table:

| PropIQ concept | Assigned-project equivalent |
|---|---|
| `listings.csv.gz` | main transaction file |
| `leads.csv` | first reference file |
| `brokers.json` | second reference file |
| `listing_id` | business key |
| `checkout_ts` | main date field |
| `status` | category/status field |
| book relationship | first genuine relationship |
| branch relationship | second genuine relationship |
| PropIQ Volume | project Volume |
| PropIQ preview table | project preview table |

Review this mapping before accepting any generated notebook.

# 22. Master conversion prompt

Copy the following prompt with all project files.

```text
Convert the attached PropIQ Week-3 notebook into a professional,
intern-facing Spark SQL notebook for my assigned data-engineering project.

AUTHORITATIVE INPUTS

1. The attached 03B notebook defines the Week-3 learning sequence,
   explanation quality, code style, evidence and Week-3 boundary.

2. The project brief defines the business context, intended entities,
   relationships and business questions.

3. The Data Pack ZIP is the physical authority for real filenames,
   folders, formats and columns.

4. The Week-2 data dictionary defines the documented grain,
   business keys and assumptions.

FIRST INSPECT — DO NOT GENERATE CODE IMMEDIATELY

Return a mapping table containing:

• actual source filename;
• file format;
• source purpose;
• one-row grain;
• business key;
• important date field;
• category or status field;
• numeric fields worth inspecting;
• reference datasets;
• genuine relationships;
• primary Bronze-preview entity.

DO NOT INVENT

• files;
• paths;
• formats;
• columns;
• data types;
• grain;
• keys;
• relationships;
• record counts;
• expected values;
• business findings.

If a required item is missing or contradictory, stop and return:

CONVERSION BLOCKED

Explain:

• the exact issue;
• the affected notebook section;
• the missing or conflicting input;
• the exact remediation required.

NOTEBOOK STYLE

The notebook is for third-year engineering interns using Databricks
for the first time.

Use Spark SQL as the main implementation language.

Also preserve a small number of short PySpark cells for:

• loading source files;
• creating DataFrames;
• creating temporary SQL views;
• displaying DataFrame rows;
• printing schemas;
• showing one or two equivalent DataFrame operations.

For selected activities, show both:

• PySpark method;
• Spark SQL method.

Clearly label what each code block is trying to do.

Use:

• %fs only to list uploaded files;
• short Python cells for file loading and DataFrame inspection;
• %sql to create temporary views;
• %sql to inspect schemas;
• %sql to display records;
• %sql for counts, distributions, checks, joins and the Bronze preview;
• one clear idea per code cell;
• short professional headings;
• context before every query;
• interpretation after every important result;
• tips and intern checkpoints;
• direct, readable code.

Do not use:

• Python helper functions or advanced Python abstractions;
• try/except;
• type annotations;
• configuration frameworks;
• path-resolution utilities;
• nested dictionaries;
• reusable ingestion frameworks;
• automated report DataFrames;
• automatic PASS/FAIL systems;
• long PySpark chains;
• advanced engineering abstractions.

PRESERVE THESE LEARNING SECTIONS

1. notebook mission and context;
2. Databricks preparation;
3. file inventory;
4. creation of one Spark SQL view per source;
5. confirmation of created views;
6. schema inspection for every source;
7. table-content display for every source;
8. grain explanation;
9. physical record counts;
10. distinct business-key count;
11. repeated-key display;
12. category or status distribution;
13. date range;
14. numeric-value range;
15. missing-value check;
16. one logical or numeric concern;
17. one timestamp or sequence concern when applicable;
18. display of suspicious records;
19. genuine relationship checks;
20. display of invalid references;
21. join-consequence explanation;
22. one simple business question;
23. one managed Delta Bronze preview;
24. source-to-preview count check;
25. preview display;
26. Week-3 versus Week-4 boundary;
27. evidence checklist;
28. final intern defence.

CONVERSION RULES

Replace all PropIQ-specific:

• names;
• file paths;
• filenames;
• formats;
• views;
• columns;
• keys;
• measurements;
• relationships;
• expected values;
• business conclusions;
• preview-table name.

Use this preview naming pattern:

workspace.default.<project_short_name>_bronze_preview_<primary_entity>

Exclude Week-10 streaming files from Week-3 work.

When a PropIQ check does not apply:

• do not invent an equivalent;
• mark it Not applicable;
• explain why;
• choose another simple, genuine check only when supported by the data.

RETURN

1. A complete project-specific IPYNB.
2. A short Databricks setup note.
3. A conversion summary.
4. A list of items interns must verify manually.
5. Any blocked or uncertain items.

The final notebook must be clean, engaging, professional,
Spark-SQL-first and implementation-ready.
```

# 23. Review the generated notebook

Before importing it into Databricks, check:

- Are the real filenames used?
- Does every column exist?
- Does the key actually identify the business entity?
- Are relationships genuine?
- Are there any invented expected values?
- Is the code mainly Spark SQL?
- Are explanations clear before and after queries?
- Are PropIQ references fully removed?

# 24. Run and verify in Databricks

Interns must:

1. import the converted notebook;
2. attach Serverless notebook compute;
3. upload the real project files;
4. update the direct Volume path;
5. run one cell at a time;
6. verify every result;
7. correct AI mistakes;
8. capture genuine screenshots;
9. complete the Week-3 log;
10. commit the verified notebook.

# 25. Remove PropIQ leftovers

Search the converted notebook for:

```text
PropIQ
propiq
listings
listing_id
leads
brokers
checkout_ts
member_code
lead_id
broker_id
90,279
90,000
558
279
```

Every PropIQ-specific reference must be removed unless the assigned project genuinely uses the same name.

# 26. AI Transparency Note

Complete:

```text
AI tool used:
Purpose:
Files provided:
What AI converted:
Files manually verified:
Columns manually verified:
Keys manually verified:
Relationships manually verified:
AI errors found:
Corrections made:
How the notebook was tested:
What every intern can explain without AI:
```

# 27. Project-conversion acceptance gate

The converted notebook is ready only when:

- [ ] all files are real;
- [ ] all columns exist;
- [ ] Spark SQL is the main language;
- [ ] code blocks remain short and readable;
- [ ] headings and explanations are professional;
- [ ] grain and keys are correct;
- [ ] values come from actual execution;
- [ ] relationships are genuine;
- [ ] suspicious records can be displayed;
- [ ] the Bronze preview reconciles;
- [ ] no PropIQ result remains;
- [ ] evidence is captured;
- [ ] every intern can explain the notebook.

# 🎉 Conversion studio complete

PropIQ provides the method.

Your Data Pack provides the truth.

AI provides a first draft.

Your team provides the engineering judgment.

```text
UNDERSTAND → CONVERT → RUN → VERIFY → CORRECT → EXPLAIN
```

> **The notebook becomes your work only after you verify it.**

# 28. Final conversion authority

The generated project notebook must include all of the following.

## Source and DataFrame foundation

- actual Volume path;
- actual source-file listing;
- one PySpark DataFrame per required Week-3 file;
- clear comments explaining each load;
- one display cell per DataFrame;
- temporary Spark SQL view creation.

## Exploration foundation

- schema inspection;
- table-content display;
- grain;
- physical row counts;
- distinct business-key counts;
- repeated-key check;
- category or status distribution;
- date range;
- numeric range;
- simple data concerns;
- genuine relationship checks;
- one business question.

## Bronze foundation

- one managed Delta Bronze table per core source;
- source columns preserved;
- `ingested_at`;
- `source_file`;
- no Silver cleaning;
- source-to-Bronze reconciliation;
- Bronze sample display;
- `DESCRIBE DETAIL`;
- `DESCRIBE HISTORY`.

## Lineage foundation

- one simple downstream exploration view built from Bronze tables;
- a visual explanation of the flow;
- Catalog Explorer lineage instructions;
- lineage screenshot requirement;
- explanation of lineage versus relationships and Delta history.

# 29. Required project-specific naming

Use simple, consistent names.

Example:

```text
workspace.default.fitpulse_bronze_activity
workspace.default.fitpulse_bronze_member
workspace.default.fitpulse_bronze_device
workspace.default.fitpulse_week03_activity_view
```

Replace `fitpulse` with the actual project short name.

Do not retain PropIQ names.

# 30. Final conversion acceptance gate

The project-specific notebook is accepted only when:

- [ ] real files are listed from the real Volume;
- [ ] every required source becomes a DataFrame;
- [ ] every DataFrame is displayed;
- [ ] every DataFrame has a temporary SQL view;
- [ ] schemas are inspected;
- [ ] SQL table content is displayed;
- [ ] grain and keys are explained;
- [ ] counts come from actual execution;
- [ ] values and ranges are explored;
- [ ] checks use real columns;
- [ ] relationships are genuine;
- [ ] one Bronze table exists per core source;
- [ ] Bronze tables include ingestion metadata;
- [ ] source and Bronze counts reconcile;
- [ ] Delta detail and history are inspected;
- [ ] one downstream exploration view demonstrates lineage;
- [ ] the Catalog Explorer lineage graph is captured;
- [ ] PropIQ references are removed;
- [ ] AI-generated assumptions are manually verified;
- [ ] every intern can defend the notebook.

# 🎉 Week-3B conversion complete

The common notebook provides the method.

The project Data Pack provides the truth.

The project team provides the judgment.

```text
INSPECT
→ MAP
→ CONVERT
→ RUN
→ VERIFY
→ CORRECT
→ EXPLAIN
```

The final output is not called “converted” merely because AI produced it.

It becomes a valid project notebook only after the team executes and verifies every section in Databricks.

# 28. Final Week-3 conversion authority

The project notebook must include real files, DataFrames, displays, temporary views, schema, grain, counts, values, concerns, relationships, one business question, exactly one Bronze demo table, one count check, Delta detail/history, one lineage demo view and one lineage screenshot.

It must not create the complete Bronze layer.

# 29. Project-specific demo naming

Example:

```text
workspace.default.fitpulse_week03_bronze_demo_activity
workspace.default.fitpulse_week03_lineage_demo_view
```

Use the main event or transaction entity. Replace `fitpulse` with the actual project short name.

# 30. Final acceptance gate

- [ ] real files used
- [ ] DataFrames created and displayed
- [ ] SQL views created and displayed
- [ ] schema, grain, keys and counts explained
- [ ] values and concerns inspected
- [ ] relationships validated
- [ ] one business question answered
- [ ] exactly one Bronze demo table created
- [ ] source and demo counts match
- [ ] Delta detail and history shown
- [ ] one lineage demo view created
- [ ] lineage screenshot captured
- [ ] no full Bronze layer created
- [ ] PropIQ names removed
- [ ] every intern can defend the work

# 🎉 Week-3B conversion complete

```text
MAP → CONVERT → RUN → VERIFY → CORRECT → EXPLAIN
```

The converted notebook is valid only after successful Databricks execution and team verification.